# 00 — Shared data preparation for Lightning AI

**Owners:** Entire team  
**Run once before all model notebooks.**

This notebook downloads only the labelled training files, performs the required
left join, creates row-level features available at prediction time, produces one
chronological 70/15/15 split, and writes the common feature audit. Every model
must use these exact partitions so the comparison is fair.

Interview explanation: *“We define the data population and holdout periods once.
Model-specific notebooks may learn different representations, but they cannot
change which transactions belong to training, validation, or final testing.”*


## Before running

1. Clone the GitHub repository into a persistent Lightning Studio.
2. Accept the IEEE-CIS competition rules on Kaggle.
3. Add the new Kaggle token as a Lightning secret named `KAGGLE_API_TOKEN`.
4. Use a CPU machine with at least 16 GB RAM; 24–32 GB is safer.

The GPU does not accelerate CSV loading or this join. Do not load the Kaggle test
files during model development—the latest 15% of labelled training data is our
honest holdout test period.


In [ ]:
from pathlib import Path
_install_root = Path.cwd().resolve()
for _candidate in [_install_root, *_install_root.parents]:
    if (_candidate / "requirements-training.txt").exists():
        _requirements = _candidate / "requirements-training.txt"
        break
else:
    raise FileNotFoundError("Open this notebook from inside the cloned repository")
%pip install -q -r {_requirements}


In [ ]:
from pathlib import Path
import gc, json, os, sys, time
import numpy as np
import pandas as pd

def locate_project_root(start=None):
    candidate = Path(start or Path.cwd()).resolve()
    for path in [candidate, *candidate.parents]:
        if (path / ".git").exists() and (path / "src").exists():
            return path
    raise FileNotFoundError("Run this notebook from inside the cloned repository")

PROJECT_ROOT = locate_project_root()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
ARTIFACT_ROOT = PROJECT_ROOT / "artifacts"
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("Project root:", PROJECT_ROOT)
print("Processed data:", PROCESSED_DIR)


## 1. Download the competition archive safely

The secret is read from the environment and never printed. Existing CSV files are
reused, making the notebook restartable.


In [ ]:
import subprocess, zipfile

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "ieee-fraud-detection"
RAW_DIR.mkdir(parents=True, exist_ok=True)
transaction_path = RAW_DIR / "train_transaction.csv"
identity_path = RAW_DIR / "train_identity.csv"

if not transaction_path.exists() or not identity_path.exists():
    if not os.getenv("KAGGLE_API_TOKEN"):
        raise RuntimeError(
            "Add KAGGLE_API_TOKEN in Lightning secrets, restart the kernel, and rerun."
        )
    subprocess.run(
        ["kaggle", "competitions", "download", "-c", "ieee-fraud-detection", "-p", str(RAW_DIR)],
        check=True,
    )
    archive = RAW_DIR / "ieee-fraud-detection.zip"
    with zipfile.ZipFile(archive) as bundle:
        bundle.extractall(RAW_DIR)

print("Transaction file:", transaction_path)
print("Identity file:", identity_path)


## 2. Load, validate, and left join

A left join retains every transaction. Only about 24.42% have an identity row;
an inner join would discard roughly three quarters of the labelled population.
Numeric columns are downcast before the join to reduce RAM usage.


In [ ]:
from src.fraud_pipeline.common import reduce_memory_usage

def memory_mb(frame):
    return frame.memory_usage(index=True, deep=True).sum() / 1024**2

transactions = reduce_memory_usage(pd.read_csv(transaction_path))
if not transactions["TransactionID"].is_unique:
    raise ValueError("train_transaction TransactionID must be unique")
raw_source_columns = [c for c in transactions.columns if c != "isFraud"]
print("Transactions:", transactions.shape, f"{memory_mb(transactions):,.1f} MB")

identities = reduce_memory_usage(pd.read_csv(identity_path))
if not identities["TransactionID"].is_unique:
    raise ValueError("train_identity TransactionID must be unique")
raw_source_columns += [c for c in identities.columns if c != "TransactionID"]
print("Identities:", identities.shape, f"{memory_mb(identities):,.1f} MB")

identity_ids = set(identities["TransactionID"].to_numpy())
transactions["has_identity"] = transactions["TransactionID"].isin(identity_ids).astype("int8")
identity_ids.clear()

joined = transactions.merge(
    identities,
    on="TransactionID",
    how="left",
    validate="one_to_one",
    sort=False,
    copy=False,
)
del transactions, identities
gc.collect()

assert len(joined) == 590_540
assert joined["TransactionID"].is_unique
assert joined["isFraud"].notna().all()
raw_dtype_map = {column: str(joined[column].dtype) for column in raw_source_columns}
print("Joined:", joined.shape, f"{memory_mb(joined):,.1f} MB")
print("Fraud rate:", f"{joined['isFraud'].mean():.4%}")
print("Identity coverage:", f"{joined['has_identity'].mean():.4%}")


## 3. Add shared, real-time-safe features

These features use only values in the current row: amount transforms,
missingness summaries, identity availability, relative time phases, and compact
card/address/email combinations. No fraud labels or future transactions are used.

`TransactionDT` has an undisclosed origin, so `transaction_relative_hour_phase`
is a periodic phase—not a claim about the real local clock or weekend.


In [ ]:
from src.fraud_pipeline.common import add_shared_features

joined = add_shared_features(joined, copy=False)
print("After shared features:", joined.shape, f"{memory_mb(joined):,.1f} MB")
display(joined[[
    "TransactionID", "TransactionAmt", "transaction_amount_log1p",
    "transaction_relative_day", "num_missing", "has_identity",
    "card_1_2", "isFraud"
]].head())


## 4. Freeze the chronological split and usable columns

Constant/all-null decisions are learned from the training period only. The final
15% is not used for feature selection, hyperparameter choice, or threshold choice.


In [ ]:
from src.fraud_pipeline.common import build_feature_audit, chronological_split

train, validation, test, split_metadata = chronological_split(joined)
del joined
gc.collect()

candidate_features = [c for c in train.columns if c not in {"isFraud", "TransactionID"}]
unusable = [
    c for c in candidate_features
    if train[c].isna().all() or train[c].nunique(dropna=False) <= 1
]
keep_columns = ["TransactionID", *[c for c in candidate_features if c not in unusable], "isFraud"]
train = train[keep_columns]
validation = validation[keep_columns]
test = test[keep_columns]

assert train["TransactionDT"].max() <= validation["TransactionDT"].min()
assert validation["TransactionDT"].max() <= test["TransactionDT"].min()
split_metadata["dropped_all_null_or_constant_from_training"] = unusable
split_metadata["model_feature_count"] = len(keep_columns) - 2

for name, frame in {"train": train, "validation": validation, "test": test}.items():
    print(name, frame.shape, f"fraud={frame['isFraud'].mean():.4%}")
print("Dropped unusable columns:", len(unusable))


## 5. Save shared Parquet data, schema, and feature audit

Parquet is substantially faster and smaller than repeatedly parsing CSV. The audit
states why each column is numeric, categorical, or identifier-like and gives the
intended representation for all four approaches.


In [ ]:
from src.fraud_pipeline.artifacts import write_json

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
train.to_parquet(PROCESSED_DIR / "train.parquet", index=False, compression="zstd")
validation.to_parquet(PROCESSED_DIR / "validation.parquet", index=False, compression="zstd")
test.to_parquet(PROCESSED_DIR / "test.parquet", index=False, compression="zstd")

audit = build_feature_audit(train)
audit.to_csv(PROCESSED_DIR / "feature_audit.csv", index=False)
write_json(PROCESSED_DIR / "split_metadata.json", split_metadata)
write_json(PROCESSED_DIR / "shared_feature_config.json", {
    "version": "1.0",
    "target": "isFraud",
    "identifier": "TransactionID",
    "time_column": "TransactionDT",
    "engineered_features": [
        c for c in train.columns
        if c not in raw_source_columns and c not in {"isFraud"}
    ],
})
write_json(PROCESSED_DIR / "raw_input_schema.json", {
    "schema_version": "1.0",
    "columns": raw_source_columns,
    "dtypes_after_memory_reduction": raw_dtype_map,
    "required_for_demo": ["TransactionDT", "TransactionAmt", "ProductCD"],
    "optional_columns": [c for c in raw_source_columns if c not in {"TransactionID", "TransactionDT", "TransactionAmt", "ProductCD"}],
    "target_is_never_an_input": "isFraud",
    "missing_optional_fields": "created as null before shared feature engineering",
})

print("Saved:")
for path in sorted(PROCESSED_DIR.iterdir()):
    print(f"  {path.name}: {path.stat().st_size / 1024**2:,.1f} MB")
display(audit.head(12))


## Completion checklist

- All 590,540 labelled transactions were retained before splitting.
- Identity coverage and fraud prevalence were validated.
- The split is chronological and shared by all teams.
- No imputer, scaler, category vocabulary, or frequency map was fitted here.
- Run notebooks `01`–`04` independently after this notebook finishes.
